# Day 20 — Split Testing
> Real-world application: testing a new email subject line.

## What is Split Testing?

Split testing (A/B testing) is a controlled experiment where:
- Users are **randomly assigned** to control (A) or treatment (B)
- A **single variable** is changed between groups
- We measure a **specific metric** (open rate, conversion, revenue)

## Minimum Sample Size

Before running a test, calculate the required sample size:
$$n = \frac{2(z_{\alpha/2} + z_\beta)^2 p(1-p)}{\delta^2}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import zt_ind_solve_power
sns.set_theme(style='whitegrid')
np.random.seed(42)

# --- Scenario: Email campaign split test ---
# Control: old subject line → expected open rate 22%
# Treatment: new subject line → expected open rate 27%

baseline_rate  = 0.22
expected_lift  = 0.05   # 5 percentage points
alpha          = 0.05
power          = 0.80

# Required sample size per group
effect_size = expected_lift / np.sqrt(baseline_rate * (1 - baseline_rate))
n_required  = zt_ind_solve_power(effect_size=effect_size, alpha=alpha, power=power, alternative='two-sided')
print(f"Required sample size per group: {int(np.ceil(n_required))}")


In [ ]:
# Simulate the experiment
n_per_group = int(np.ceil(n_required))
control_opens   = np.random.binomial(1, 0.22, n_per_group)
treatment_opens = np.random.binomial(1, 0.27, n_per_group)

rate_c = control_opens.mean()
rate_t = treatment_opens.mean()
print(f"Control open rate   : {rate_c:.3%}")
print(f"Treatment open rate : {rate_t:.3%}")
print(f"Absolute lift       : {rate_t - rate_c:+.3%}")
print(f"Relative lift       : {(rate_t - rate_c)/rate_c:+.1%}")


In [ ]:
# Statistical test (two-proportion z-test)
from statsmodels.stats.proportion import proportions_ztest
count = np.array([treatment_opens.sum(), control_opens.sum()])
nobs  = np.array([n_per_group, n_per_group])
z_stat, p_value = proportions_ztest(count, nobs)

print(f"\nTwo-proportion z-test")
print(f"z = {z_stat:.4f}, p = {p_value:.6f}")
print(f"Decision: {'Reject H₀ — new subject line performs better' if p_value<alpha else 'No significant difference'}")

# Confidence interval for the difference
from statsmodels.stats.proportion import proportion_confint
ci_c = proportion_confint(control_opens.sum(),   n_per_group, alpha=0.05)
ci_t = proportion_confint(treatment_opens.sum(), n_per_group, alpha=0.05)
print(f"\nControl CI  : ({ci_c[0]:.3%}, {ci_c[1]:.3%})")
print(f"Treatment CI: ({ci_t[0]:.3%}, {ci_t[1]:.3%})")


In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart with CI
groups = ['Control', 'Treatment']
rates  = [rate_c, rate_t]
colors = ['#4C72B0', '#DD8452']
ci_lows  = [rate_c - ci_c[0],  rate_t - ci_t[0]]
ci_highs = [ci_c[1] - rate_c,  ci_t[1] - rate_t]

for i, (g, r, cl, ch, col) in enumerate(zip(groups, rates, ci_lows, ci_highs, colors)):
    axes[0].bar(i, r, color=col, alpha=0.7, width=0.4)
    axes[0].errorbar(i, r, yerr=[[cl],[ch]], fmt='none', color='black', capsize=10, capthick=2)
    axes[0].text(i, r + ch + 0.005, f'{r:.2%}', ha='center', fontweight='bold')

axes[0].set_xticks([0,1]); axes[0].set_xticklabels(groups)
axes[0].set_ylabel('Open Rate'); axes[0].set_title('Email Open Rate by Variant')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0%}'))

# Distribution of proportions under H0
x = np.linspace(-4, 4, 400)
y = stats.norm.pdf(x)
crit = stats.norm.ppf(0.975)
axes[1].plot(x, y, 'k-', lw=2)
axes[1].fill_between(x, y, where=(x>=crit)|(x<=-crit), alpha=0.3, color='red', label='α=0.05')
axes[1].axvline(z_stat, color='navy', lw=2, linestyle='--', label=f'z={z_stat:.2f} (p={p_value:.4f})')
axes[1].set_title('Test Statistic on Normal Distribution')
axes[1].legend()

plt.suptitle('Split Test: Email Subject Line Experiment', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/06_split_testing.png', dpi=150)
plt.show()
